In [ ]:
05-03-2026

# 1. Clonar o repositório 
git clone https://github.com/{illanademiranda}/senac_ia_uc15_nlp_project.git

# 2. Entrar na pasta do projeto
cd senac_ia_uc15_nlp_project

# 3. Criar o Ambiente Virtual (essencial para não dar conflito de bibliotecas)
python -m venv venv

# 4. Ativar o ambiente (No Windows)
.\venv\Scripts\activate

# 5. Criar o arquivo de dependências inicial
echo pypdf > requirements.txt
echo langchain >> requirements.txt

# Ambientes virtuais
venv/
.env

# Arquivos temporários do Python
__pycache__/
*.py[cod]

# Arquivos do Sistema Operacional
.DS_Store
Thumbs.db

In [ ]:
06-03-2026

import re
import unicodedata

def limpar_texto_base(texto):
    """
    Função para sanitizar o texto bruto extraído de fontes públicas ou PDFs.
    Remove espaços duplos, caracteres especiais desnecessários e normaliza o texto.
    """
    if not texto:
        return ""

    # 1. Normalização Unicode (trata acentos e caracteres especiais)
    texto = unicodedata.normalize('NFKC', texto)

    # 2. Remover quebras de linha excessivas e transforma em espaço simples
    texto = re.sub(r'\s+', ' ', texto)

    # 3. Remover caracteres que não são úteis para o modelo NLP (opcional, dependendo da base)
    # Aqui mantemos letras, números e pontuação básica
    texto = re.sub(r'[^\w\s.,!?;:-]', '', texto)

    return texto.strip()

# Teste inicial de validação do dia 06/03
exemplo_sujo = "Convenção  Coletiva 2025\n\n  - Salário: R$ 2.500,00!!! @@@"
print(f"Original: {exemplo_sujo}")
print(f"Limpo: {limpar_texto_base(exemplo_sujo)}")

In [ ]:
09-03-2026

from pypdf import PdfReader

def extrair_texto_com_metadados(caminho_pdf):
    """
    Extrai o texto do PDF mantendo a referência da página.
    Isso é crucial para o 'Gold Standard' e para as citações do bot.
    """
    reader = PdfReader(caminho_pdf)
    documento_estruturado = []

    for i, pagina in enumerate(reader.pages):
        texto_bruto = pagina.extract_text()
        
        # Usar a limpeza criada no dia 06/03
        texto_limpo = " ".join(texto_bruto.split()) 
        
        if texto_limpo:
            # Criar um dicionário que vincula o conteúdo à página
            documento_estruturado.append({
                "pagina": i + 1,
                "conteudo": texto_limpo
            })
            
    return documento_estruturado

# Caminho do arquivo que o Marcelo subiu
caminho = "data/sindilojas_2025_2026.pdf"

try:
    dados_extraidos = extrair_texto_com_metadados(caminho)
    print(f"Sucesso! {len(dados_extraidos)} páginas processadas.")
    
    # Exemplo de como isso ajuda no Gold Standard:
    print(f"Conteúdo da Página 1: {dados_extraidos[0]['conteudo'][:100]}...")
except Exception as e:
    print(f"Erro ao ler PDF: {e}")

In [ ]:
10-03-2026
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Carregamento com Metadados (O que você defendeu no log)
loader = PyPDFLoader("data/sindilojas_2025_2026.pdf")
paginas = loader.load()

# 2. Divisão de Texto (Text Splitting)
# O modelo não consegue ler o PDF inteiro de uma vez, precisamos de "pedaços" (chunks)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)
documentos_divididos = text_splitter.split_documents(paginas)

# 3. Criação de Embeddings (Representação matemática do texto)
# Usando um modelo gratuito do HuggingFace (conforme o log do Arthur)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Criação do Vector Database (FAISS - rápido e eficiente para local)
# Isso permite que a IA "busque" o trecho certo antes de responder
vector_db = FAISS.from_documents(documentos_divididos, embeddings)

# 5. Teste de Busca (Simulando uma pergunta do Gold Standard)
query = "Qual o piso salarial do vendedor?"
resultado = vector_db.similarity_search(query, k=2)

print(f"Trecho mais relevante encontrado na página {resultado[0].metadata['page'] + 1}:")
print(resultado[0].page_content)

In [ ]:
11-03-2026
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Carregamento com Preservação de Metadados
# O PyPDFLoader automaticamente anota de qual página cada trecho veio
loader = PyPDFLoader("data/sindilojas_2025_2026.pdf")
documentos_brutos = loader.load()

# 2. Estratégia de Chunking (Divisão do texto)
# Usamos o RecursiveCharacterTextSplitter por ser mais inteligente na divisão
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # Tamanho do pedaço de texto
    chunk_overlap=150,    # Sobreposição para não perder contexto entre pedaços
    separators=["\n\n", "\n", ".", " ", ""]
)
documentos_finais = text_splitter.split_documents(documentos_brutos)

# 3. Geração de Embeddings
# Modelo leve e eficiente do HuggingFace para rodar localmente
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# 4. Criação da Base Vetorial (Vector Store)
# FAISS é uma biblioteca do Facebook para busca rápida em vetores
vector_db = FAISS.from_documents(documentos_finais, embeddings)

# 5. Teste de Recuperação (Simulando uma pergunta do Gold Standard)
query = "Qual a regra para trabalho em feriados?"
docs_recuperados = vector_db.similarity_search(query, k=2)

print(f"Foram gerados {len(documentos_finais)} chunks.")
print(f"Top 1 resultado (Página {docs_recuperados[0].metadata['page'] + 1}):")
print(docs_recuperados[0].page_content)

In [ ]:
12-03-2026
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 1. Carregar o Modelo para comparação semântica (ver se o sentido é o mesmo)
model_aval = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Definir o seu "Gold Standard" (Exemplos baseados no seu arquivo .md)
# Aqui você deve colocar as perguntas e respostas reais do seu PDF
test_cases = [
    {
        "pergunta": "Qual o piso salarial do vendedor?",
        "gold_answer": "O piso salarial para a categoria de vendedores é de R$ 1.500,00 conforme cláusula 3ª.",
    },
    {
        "pergunta": "Qual o adicional de horas extras em feriados?",
        "gold_answer": "As horas extras trabalhadas em feriados terão acréscimo de 100% sobre a hora normal.",
    }
    # Adicione as outras 8 aqui...
]

def avaliar_sistema(vector_db):
    resultados = []
    
    for case in test_cases:
        # A IA busca a resposta no PDF
        docs = vector_db.similarity_search(case["pergunta"], k=1)
        ai_answer = docs[0].page_content
        
        # Comparação semântica (Cosseno de Similaridade)
        # Verifica se o significado da resposta da IA bate com o Gold Standard
        emb_gold = model_aval.encode(case["gold_answer"], convert_to_tensor=True)
        emb_ai = model_aval.encode(ai_answer, convert_to_tensor=True)
        similaridade = util.pytorch_cos_sim(emb_gold, emb_ai).item()
        
        resultados.append({
            "Pergunta": case["pergunta"],
            "Similaridade": round(similaridade, 4),
            "Status": "✅ Aprovado" if similaridade > 0.7 else "❌ Revisar"
        })
    
    return pd.DataFrame(resultados)

# Simulação de execução
# df_validacao = avaliar_sistema(vector_db) # vector_db vem do script do dia 11
# print(df_validacao)